In [118]:
from dataclasses import dataclass
from typing import Optional, Dict, List, Any

import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)

from sklearn.model_selection import train_test_split


# ============================================================
# Optional survival packages
# ============================================================

try:
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index

    LIFELINES_AVAILABLE = True

except ImportError:
    LIFELINES_AVAILABLE = False


try:
    from sksurv.ensemble import RandomSurvivalForest
    from sksurv.util import Surv
    from sksurv.metrics import concordance_index_censored

    SKSURV_AVAILABLE = True

except ImportError:
    SKSURV_AVAILABLE = False

In [119]:



# ============================================================
# Configuration
# ============================================================

@dataclass
class ModelRunnerConfig:

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    classification_horizon_days: int = 365

    classification_threshold: float = 0.5

    # --------------------------------------------------------
    # Train / test split
    # --------------------------------------------------------

    split_mode: str = "temporal"
    test_size: float = 0.20

    temporal_column: str = "snapshot_date"

    random_state: int = 42

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------

    logistic_max_iter: int = 3000

    logistic_class_weight: Optional[str] = "balanced"

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    rf_n_estimators: int = 400

    rf_max_depth: Optional[int] = None

    rf_min_samples_leaf: int = 5

    rf_class_weight: Optional[str] = "balanced"

    # --------------------------------------------------------
    # Random Survival Forest
    # --------------------------------------------------------

    rsf_n_estimators: int = 400

    rsf_min_samples_split: int = 10

    rsf_min_samples_leaf: int = 5

    rsf_max_features: str = "sqrt"

    # --------------------------------------------------------
    # Modules
    # --------------------------------------------------------

    include_modules: Optional[List[str]] = None

    # --------------------------------------------------------
    # Targets
    # --------------------------------------------------------

    duration_column: str = "target_remaining_days"

    event_column: str = "target_event"


# ============================================================
# Utility
# ============================================================

def _clean_feature_matrix(
    df: pd.DataFrame,
    features: List[str]
) -> pd.DataFrame:
    """
    Selects features and forces values to numeric.

    Intended for an already encoded snapshot dataset.
    """

    available = [
        feature
        for feature in features
        if feature in df.columns
    ]

    X = df[available].copy()

    for column in X.columns:

        if X[column].dtype == bool:
            X[column] = X[column].astype(int)

        else:
            X[column] = pd.to_numeric(
                X[column],
                errors="coerce"
            )

    # Remove completely empty features
    X = X.dropna(
        axis=1,
        how="all"
    )

    # Remove constant features
    constant_columns = [
        c
        for c in X.columns
        if X[c].nunique(dropna=True) <= 1
    ]

    X = X.drop(
        columns=constant_columns,
        errors="ignore"
    )

    return X


# ============================================================
# Split function
# ============================================================

def create_train_test_indices(
    df: pd.DataFrame,
    config: ModelRunnerConfig
):

    if config.split_mode == "temporal":

        if config.temporal_column not in df.columns:
            raise ValueError(
                f"Temporal column "
                f"'{config.temporal_column}' not found."
            )

        dates = pd.to_datetime(
            df[config.temporal_column],
            errors="coerce"
        )

        valid = dates.notna()

        ordered_indices = (
            df.loc[valid]
            .assign(_date=dates[valid])
            .sort_values("_date")
            .index
            .to_numpy()
        )

        split_point = int(
            len(ordered_indices)
            *
            (1 - config.test_size)
        )

        train_idx = ordered_indices[:split_point]

        test_idx = ordered_indices[split_point:]

        return train_idx, test_idx

    elif config.split_mode == "random":

        indices = df.index.to_numpy()

        train_idx, test_idx = train_test_split(
            indices,
            test_size=config.test_size,
            random_state=config.random_state
        )

        return train_idx, test_idx

    else:

        raise ValueError(
            "split_mode must be 'temporal' or 'random'."
        )


# ============================================================
# Classification target
# ============================================================

def create_horizon_classification_data(
    df: pd.DataFrame,
    horizon_days: int,
    duration_col: str,
    event_col: str
):
    """
    Creates a binary outcome:

        1 = turnover observed within horizon
        0 = known to survive beyond horizon

    IMPORTANT:

    Cases censored BEFORE the horizon are removed because their
    true classification outcome is unknown.
    """

    duration = pd.to_numeric(
        df[duration_col],
        errors="coerce"
    )

    event = pd.to_numeric(
        df[event_col],
        errors="coerce"
    )

    # Known turnover before horizon
    positive = (
        (event == 1)
        &
        (duration <= horizon_days)
    )

    # Known survival beyond horizon
    negative = (
        duration > horizon_days
    )

    valid = positive | negative

    y = pd.Series(
        np.nan,
        index=df.index
    )

    y.loc[positive] = 1
    y.loc[negative] = 0

    return valid, y.astype("Int64")


# ============================================================
# Classification preprocessing
# ============================================================

def make_logistic_pipeline(
    config: ModelRunnerConfig
):

    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=config.logistic_max_iter,
                    class_weight=config.logistic_class_weight,
                    random_state=config.random_state
                )
            )
        ]
    )


def make_random_forest_pipeline(
    config: ModelRunnerConfig
):

    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=config.rf_n_estimators,
                    max_depth=config.rf_max_depth,
                    min_samples_leaf=config.rf_min_samples_leaf,
                    class_weight=config.rf_class_weight,
                    random_state=config.random_state,
                    n_jobs=-1
                )
            )
        ]
    )


# ============================================================
# Classification evaluation
# ============================================================

def evaluate_classifier(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    threshold=0.5
):

    model.fit(
        X_train,
        y_train
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    result = {}

    # --------------------------------------------------------
    # Metrics requiring both classes
    # --------------------------------------------------------

    if len(np.unique(y_test)) > 1:

        result["roc_auc"] = roc_auc_score(
            y_test,
            probabilities
        )

        result["pr_auc"] = average_precision_score(
            y_test,
            probabilities
        )

    else:

        result["roc_auc"] = np.nan
        result["pr_auc"] = np.nan

    # --------------------------------------------------------

    result["precision"] = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    result["recall"] = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    result["f1"] = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    result["brier"] = brier_score_loss(
        y_test,
        probabilities
    )

    result["n_train"] = len(y_train)

    result["n_test"] = len(y_test)

    result["events_train"] = int(
        y_train.sum()
    )

    result["events_test"] = int(
        y_test.sum()
    )

    return model, result


# ============================================================
# Cox PH
# ============================================================

def fit_cox_model(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: List[str],
    duration_col: str,
    event_col: str
):

    if not LIFELINES_AVAILABLE:

        raise ImportError(
            "lifelines is required for Cox PH.\n"
            "Install with:\n"
            "pip install lifelines"
        )

    X_train = _clean_feature_matrix(
        train_df,
        features
    )

    X_test = _clean_feature_matrix(
        test_df,
        list(X_train.columns)
    )

    # --------------------------------------------------------
    # Median imputation
    # --------------------------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    train_values = imputer.fit_transform(
        X_train
    )

    test_values = imputer.transform(
        X_test
    )

    X_train_imp = pd.DataFrame(
        train_values,
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_imp = pd.DataFrame(
        test_values,
        columns=X_train.columns,
        index=X_test.index
    )

    # --------------------------------------------------------
    # Standardize for numerical stability
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(
            X_train_imp
        ),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_scaled = pd.DataFrame(
        scaler.transform(
            X_test_imp
        ),
        columns=X_train.columns,
        index=X_test.index
    )

    train_cox = X_train_scaled.copy()

    train_cox[duration_col] = (
        train_df.loc[
            X_train.index,
            duration_col
        ]
    )

    train_cox[event_col] = (
        train_df.loc[
            X_train.index,
            event_col
        ]
    )

    # --------------------------------------------------------
    # Penalizer adds numerical stability with many features
    # --------------------------------------------------------

    cox = CoxPHFitter(
        penalizer=0.01
    )

    cox.fit(
        train_cox,
        duration_col=duration_col,
        event_col=event_col
    )

    # --------------------------------------------------------
    # Predicted partial hazard
    # --------------------------------------------------------

    risk = (
        cox.predict_partial_hazard(
            X_test_scaled
        )
        .values
        .reshape(-1)
    )

    durations = (
        test_df.loc[
            X_test.index,
            duration_col
        ]
        .astype(float)
        .values
    )

    events = (
        test_df.loc[
            X_test.index,
            event_col
        ]
        .astype(int)
        .values
    )

    # lifelines concordance expects higher predicted score
    # to imply LONGER survival.
    # Partial hazard means the opposite -> negative sign.
    c_index = concordance_index(
        durations,
        -risk,
        events
    )

    result = {

        "c_index": c_index,

        "n_train": len(train_cox),

        "n_test": len(X_test_scaled),

        "events_train": int(
            train_cox[event_col].sum()
        ),

        "events_test": int(
            events.sum()
        )
    }

    preprocessing = {
        "features": list(X_train.columns),
        "imputer": imputer,
        "scaler": scaler
    }

    return (
        cox,
        result,
        preprocessing
    )


# ============================================================
# Random Survival Forest
# ============================================================

def fit_random_survival_forest(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: List[str],
    duration_col: str,
    event_col: str,
    config: ModelRunnerConfig
):

    if not SKSURV_AVAILABLE:

        raise ImportError(
            "scikit-survival is required for RSF.\n"
            "Install with:\n"
            "pip install scikit-survival"
        )

    X_train = _clean_feature_matrix(
        train_df,
        features
    )

    X_test = _clean_feature_matrix(
        test_df,
        list(X_train.columns)
    )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_train.columns,
        index=X_test.index
    )

    y_train = Surv.from_arrays(
        event=train_df.loc[
            X_train.index,
            event_col
        ].astype(bool),

        time=train_df.loc[
            X_train.index,
            duration_col
        ].astype(float)
    )

    y_test = Surv.from_arrays(
        event=test_df.loc[
            X_test.index,
            event_col
        ].astype(bool),

        time=test_df.loc[
            X_test.index,
            duration_col
        ].astype(float)
    )

    model = RandomSurvivalForest(
        n_estimators=config.rsf_n_estimators,
        min_samples_split=config.rsf_min_samples_split,
        min_samples_leaf=config.rsf_min_samples_leaf,
        max_features=config.rsf_max_features,
        n_jobs=-1,
        random_state=config.random_state
    )

    model.fit(
        X_train_imp,
        y_train
    )

    risk = model.predict(
        X_test_imp
    )

    c_index = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk
    )[0]

    result = {

        "c_index": c_index,

        "n_train": len(X_train_imp),

        "n_test": len(X_test_imp),

        "events_train": int(
            y_train["event"].sum()
        ),

        "events_test": int(
            y_test["event"].sum()
        )
    }

    preprocessing = {
        "features": list(X_train.columns),
        "imputer": imputer
    }

    return (
        model,
        result,
        preprocessing
    )


# ============================================================
# Main model runner
# ============================================================

def run_models(
    snapshot_df: pd.DataFrame,
    modules: Dict[str, List[str]],
    config: Optional[ModelRunnerConfig] = None
):
    """
    Runs all selected models over every selected feature module.

    Returns:

        {
            "results": pd.DataFrame,
            "models": {...},
            "preprocessing": {...},
            "cox_summaries": {...},
            "train_indices": ...,
            "test_indices": ...
        }
    """

    if config is None:
        config = ModelRunnerConfig()

    df = snapshot_df.copy()

    # --------------------------------------------------------
    # Basic target cleaning
    # --------------------------------------------------------

    df[config.duration_column] = pd.to_numeric(
        df[config.duration_column],
        errors="coerce"
    )

    df[config.event_column] = pd.to_numeric(
        df[config.event_column],
        errors="coerce"
    )

    df = df[
        df[config.duration_column].notna()
        &
        df[config.event_column].notna()
        &
        (df[config.duration_column] >= 0)
    ].copy()

    # --------------------------------------------------------
    # Train/test split
    # --------------------------------------------------------

    train_idx, test_idx = create_train_test_indices(
        df,
        config
    )

    train_df = df.loc[
        train_idx
    ].copy()

    test_df = df.loc[
        test_idx
    ].copy()

    # --------------------------------------------------------
    # Module selection
    # --------------------------------------------------------

    if config.include_modules is None:

        module_names = list(
            modules.keys()
        )

    else:

        module_names = [
            m
            for m in config.include_modules
            if m in modules
        ]

    results = []

    fitted_models = {}

    preprocessing_objects = {}

    cox_summaries = {}

    # ========================================================
    # Iterate modules
    # ========================================================

    for module_name in module_names:

        features = modules[
            module_name
        ]

        if not features:

            warnings.warn(
                f"Skipping empty module "
                f"'{module_name}'."
            )

            continue

        print(
            f"\n"
            f"====================================\n"
            f"MODULE: {module_name}\n"
            f"Features: {len(features)}\n"
            f"===================================="
        )

        # ====================================================
        # CLASSIFICATION DATA
        # ====================================================

        train_valid, train_y_all = (
            create_horizon_classification_data(
                train_df,
                config.classification_horizon_days,
                config.duration_column,
                config.event_column
            )
        )

        test_valid, test_y_all = (
            create_horizon_classification_data(
                test_df,
                config.classification_horizon_days,
                config.duration_column,
                config.event_column
            )
        )

        classification_train = train_df.loc[
            train_valid
        ]

        classification_test = test_df.loc[
            test_valid
        ]

        X_train_cls = _clean_feature_matrix(
            classification_train,
            features
        )

        X_test_cls = _clean_feature_matrix(
            classification_test,
            list(X_train_cls.columns)
        )

        # Align test features exactly
        X_test_cls = X_test_cls.reindex(
            columns=X_train_cls.columns
        )

        y_train_cls = (
            train_y_all.loc[
                X_train_cls.index
            ]
            .astype(int)
        )

        y_test_cls = (
            test_y_all.loc[
                X_test_cls.index
            ]
            .astype(int)
        )

        # ====================================================
        # LOGISTIC REGRESSION
        # ====================================================

        if (
            len(X_train_cls) > 0
            and
            y_train_cls.nunique() > 1
        ):

            try:

                logistic = (
                    make_logistic_pipeline(
                        config
                    )
                )

                logistic, metrics = (
                    evaluate_classifier(
                        logistic,
                        X_train_cls,
                        y_train_cls,
                        X_test_cls,
                        y_test_cls,
                        threshold=(
                            config
                            .classification_threshold
                        )
                    )
                )

                result = {
                    "module": module_name,
                    "model": "Logistic Regression",
                    "model_type": "classification",
                    "n_features": len(
                        X_train_cls.columns
                    ),
                    "horizon_days": (
                        config
                        .classification_horizon_days
                    ),
                    **metrics
                }

                results.append(result)

                fitted_models[
                    (
                        module_name,
                        "logistic"
                    )
                ] = logistic

            except Exception as e:

                warnings.warn(
                    f"Logistic Regression failed "
                    f"for {module_name}: {e}"
                )

        # ====================================================
        # RANDOM FOREST CLASSIFIER
        # ====================================================

        if (
            len(X_train_cls) > 0
            and
            y_train_cls.nunique() > 1
        ):

            try:

                rf = (
                    make_random_forest_pipeline(
                        config
                    )
                )

                rf, metrics = (
                    evaluate_classifier(
                        rf,
                        X_train_cls,
                        y_train_cls,
                        X_test_cls,
                        y_test_cls,
                        threshold=(
                            config
                            .classification_threshold
                        )
                    )
                )

                result = {
                    "module": module_name,
                    "model": "Random Forest",
                    "model_type": "classification",
                    "n_features": len(
                        X_train_cls.columns
                    ),
                    "horizon_days": (
                        config
                        .classification_horizon_days
                    ),
                    **metrics
                }

                results.append(result)

                fitted_models[
                    (
                        module_name,
                        "random_forest"
                    )
                ] = rf

            except Exception as e:

                warnings.warn(
                    f"Random Forest failed "
                    f"for {module_name}: {e}"
                )

        # ====================================================
        # COX PH
        # ====================================================

        try:

            (
                cox,
                cox_metrics,
                cox_preprocessing
            ) = fit_cox_model(
                train_df,
                test_df,
                features,
                config.duration_column,
                config.event_column
            )

            result = {
                "module": module_name,
                "model": "Cox PH",
                "model_type": "survival",
                "n_features": len(
                    cox_preprocessing[
                        "features"
                    ]
                ),
                "horizon_days": np.nan,
                **cox_metrics
            }

            results.append(result)

            fitted_models[
                (
                    module_name,
                    "cox"
                )
            ] = cox

            preprocessing_objects[
                (
                    module_name,
                    "cox"
                )
            ] = cox_preprocessing

            cox_summaries[
                module_name
            ] = cox.summary.copy()

        except Exception as e:

            warnings.warn(
                f"Cox PH failed "
                f"for {module_name}: {e}"
            )

        # ====================================================
        # RANDOM SURVIVAL FOREST
        # ====================================================

        try:

            (
                rsf,
                rsf_metrics,
                rsf_preprocessing
            ) = fit_random_survival_forest(
                train_df,
                test_df,
                features,
                config.duration_column,
                config.event_column,
                config
            )

            result = {
                "module": module_name,
                "model": "Random Survival Forest",
                "model_type": "survival",
                "n_features": len(
                    rsf_preprocessing[
                        "features"
                    ]
                ),
                "horizon_days": np.nan,
                **rsf_metrics
            }

            results.append(result)

            fitted_models[
                (
                    module_name,
                    "rsf"
                )
            ] = rsf

            preprocessing_objects[
                (
                    module_name,
                    "rsf"
                )
            ] = rsf_preprocessing

        except Exception as e:

            warnings.warn(
                f"Random Survival Forest failed "
                f"for {module_name}: {e}"
            )

    # ========================================================
    # Result table
    # ========================================================

    results_df = pd.DataFrame(
        results
    )

    if not results_df.empty:

        preferred_order = [
            "module",
            "model",
            "model_type",
            "n_features",
            "n_train",
            "n_test",
            "events_train",
            "events_test",
            "horizon_days",
            "roc_auc",
            "pr_auc",
            "precision",
            "recall",
            "f1",
            "brier",
            "c_index"
        ]

        existing_columns = [
            c
            for c in preferred_order
            if c in results_df.columns
        ]

        remaining_columns = [
            c
            for c in results_df.columns
            if c not in existing_columns
        ]

        results_df = results_df[
            existing_columns
            +
            remaining_columns
        ]

    return {

        "results":
            results_df,

        "models":
            fitted_models,

        "preprocessing":
            preprocessing_objects,

        "cox_summaries":
            cox_summaries,

        "train_indices":
            train_idx,

        "test_indices":
            test_idx,

        "config":
            config
    }

In [120]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ModuleConfig:
    demographic: List[str] = field(default_factory=list)
    experience: List[str] = field(default_factory=list)
    organisational: List[str] = field(default_factory=list)

    include_work_history: bool = True
    include_client_exposure: bool = True
    include_assignment_stability: bool = True
    include_temporal_dynamics: bool = True
    include_caregiver_client_fit: bool = True
    include_package_exposure: bool = False

In [121]:



def build_feature_modules(
    snapshot_df: pd.DataFrame,
    config: ModuleConfig
):
    """
    Build interpretable predictor modules from the flattened caregiver snapshot.
    """

    modules = {}

    modules["demographic"] = [
        c for c in config.demographic
        if c in snapshot_df.columns
    ]

    modules["experience"] = [
        c for c in config.experience
        if c in snapshot_df.columns
    ]

    modules["organisational"] = [
        c for c in config.organisational
        if c in snapshot_df.columns
    ]

    # --------------------------------------------------------
    # Work history
    # --------------------------------------------------------
    if config.include_work_history:
        candidates = [
            "days_since_recruitment",
            "number_assignments",
            "number_unique_clients",
            "workdays",
            "offworkdays",
            "work_ratio",
            "mean_assignment_length",
            "median_assignment_length",
            "min_assignment_length",
            "max_assignment_length",
            "std_assignment_length",
            "mean_offwork_length",
            "median_offwork_length",
            "max_offwork_length",
            "days_since_last_assignment",
        ]

        modules["work_history"] = [
            c for c in candidates
            if c in snapshot_df.columns
        ]
    else:
        modules["work_history"] = []

    # --------------------------------------------------------
    # Client exposure
    # --------------------------------------------------------
    if config.include_client_exposure:
        modules["client_exposure"] = [
            c for c in snapshot_df.columns
            if (
                c.startswith("days_client_class_")
                or c.startswith("share_client_class_")
                or c in [
                    "average_client_class",
                    "classified_client_days",
                ]
            )
        ]
    else:
        modules["client_exposure"] = []

    # --------------------------------------------------------
    # Assignment stability
    # --------------------------------------------------------
    if config.include_assignment_stability:
        candidates = [
            "repeat_client_ratio",
            "current_client_class",
            "previous_client_class",
            "client_class_change",
        ]

        modules["assignment_stability"] = [
            c for c in candidates
            if c in snapshot_df.columns
        ]
    else:
        modules["assignment_stability"] = []

    # --------------------------------------------------------
    # Temporal dynamics
    # --------------------------------------------------------
    if config.include_temporal_dynamics:
        explicit_temporal = [
            "last_assignment_length",
            "mean_assignment_length_last3",
            "std_assignment_length_last3",
            "assignment_length_trend",
            "last_assignment_vs_historical_mean",
            "last_offwork_length",
            "mean_offwork_length_last3",
            "std_offwork_length_last3",
            "offwork_gap_trend",
            "last_gap_vs_historical_mean",
            "current_gap_ratio_to_historical",
        ]

        rolling_temporal = [
            c for c in snapshot_df.columns
            if (
                c.startswith("workdays_last_")
                or c.startswith("offworkdays_last_")
                or c.startswith("work_ratio_last_")
                or c.startswith("assignments_last_")
            )
        ]

        modules["temporal_dynamics"] = [
            c for c in dict.fromkeys(
                explicit_temporal + rolling_temporal
            )
            if c in snapshot_df.columns
        ]
    else:
        modules["temporal_dynamics"] = []

    # --------------------------------------------------------
    # Caregiver-client fit
    # --------------------------------------------------------
    if config.include_caregiver_client_fit:
        explicit_fit = [
            "requirement_exposure_days",
            "matched_requirement_days",
            "unmatched_requirement_days",
            "caregiver_client_match_ratio",
            "caregiver_client_mismatch_ratio",
            "requirement_type_match_ratio",
            "requirement_types_seen",
            "requirement_types_matched",
        ]

        mismatch_features = [
            c for c in snapshot_df.columns
            if c.startswith("mismatch_days_")
        ]

        modules["caregiver_client_fit"] = [
            c for c in dict.fromkeys(
                explicit_fit + mismatch_features
            )
            if c in snapshot_df.columns
        ]
    else:
        modules["caregiver_client_fit"] = []

    # --------------------------------------------------------
    # Optional package exposure
    # --------------------------------------------------------
    if config.include_package_exposure:
        modules["package_exposure"] = [
            c for c in snapshot_df.columns
            if (
                c.startswith("days_package_")
                or c.startswith("share_package_")
            )
        ]
    else:
        modules["package_exposure"] = []

    # Complete model = union of all active modules
    complete = []
    for name, cols in modules.items():
        if name == "complete":
            continue
        complete.extend(cols)

    modules["complete"] = list(dict.fromkeys(complete))

    return modules


# Example module configuration.
# Adjust the explicit static feature lists to the columns you want to retain.
module_config = ModuleConfig(
    demographic=[
        "age_at_snapshot",
        "gender_M",
        "nationality_Rumänien",
        "nationality_Ungarn",
        "nationality_Slowakei",
        "nationality_Polen",
        "nationality_Österreich",
    ],

    experience=[
        "ausbildung-krankenschwester",
        "erfahrung-krankenhaus",
        "erfahrung-alzheimer",
        "erfahrung-demenz-beginnend",
        "erfahrung-demenz-aggressiv",
        "erfahrung-schlaganfall",
        "erfahrung-parkinson",
        "erfahrung-copd",
        "erfahrung-mobilitaet-rollstuhl",
        "erfahrung-bettspflege",
        "erfahrung-diabetes",
        "erfahrung-palliativ",
    ],

    organisational=[
        "geschBereich_Geschäftsbereich H24",
        "geschBereich_Geschäftsbereich PML",
    ],

    include_work_history=True,
    include_client_exposure=True,
    include_assignment_stability=True,
    include_temporal_dynamics=True,
    include_caregiver_client_fit=True,
    include_package_exposure=False,
)

modules = build_feature_modules(
    snapshot_df,
    module_config
)

for module_name, feature_names in modules.items():
    print(
        f"{module_name:25s}: "
        f"{len(feature_names):3d} features"
    )


demographic              :   7 features
experience               :  12 features
organisational           :   2 features
work_history             :  15 features
client_exposure          :  10 features
assignment_stability     :   4 features
temporal_dynamics        :  23 features
caregiver_client_fit     :  30 features
package_exposure         :   0 features
complete                 : 103 features


In [122]:
module_config = ModuleConfig(
    demographic=[
        "age_at_snapshot",
        "gender_M",
        "nationality_Rumänien",
        "nationality_Ungarn",
        "nationality_Slowakei",
        "nationality_Polen",
        "nationality_Österreich",
    ],

    experience=[
        "ausbildung-krankenschwester",
        "erfahrung-krankenhaus",
        "erfahrung-alzheimer",
        "erfahrung-demenz-beginnend",
        "erfahrung-demenz-aggressiv",
        "erfahrung-schlaganfall",
        "erfahrung-parkinson",
        "erfahrung-copd",
        "erfahrung-mobilitaet-rollstuhl",
        "erfahrung-bettspflege",
        "erfahrung-diabetes",
        "erfahrung-palliativ",
    ],

    organisational=[
        "geschBereich_Geschäftsbereich H24",
        "geschBereich_Geschäftsbereich PML",
    ],

    include_work_history=True,
    include_client_exposure=True,
    include_assignment_stability=True,
    include_temporal_dynamics=True,
    include_caregiver_client_fit=True,
    include_package_exposure=False,
)

In [138]:
snapshot_df = pd.read_csv("snapshot_cleaned.csv")

snapshot_df


,eingestellt-am,ausgesch-am,dateOfBirth,Betreuer:innen_Austritt_Ja,org_Team Polen,nationality_Deutschland,nationality_Österreich,snapshot_date,recruitment_date_clean,exit_date_clean,...,share_package_Demenzbetreuung,share_package_Diplomierte_Fachkraft,share_package_KH_Nachversorgung,share_package_Klassik,share_package_Palliativbegleitung,share_package_Premium,share_package_Sonstige,share_package_ZZ_Pflege_leicht,share_package_ZZ_Pflegerin,share_package_ZZ_Standard_Plus
0,20230613.0,20260120.0,19850117,0,0,0,0,2025-01-08,2023-06-13,2026-01-20,...,0.0,0.0,0.0,0.455422,0.0,0.342169,0.0,0.0,0.0,0.000000
1,20220914.0,20241203.0,19821104,0,0,0,0,2024-05-16,2022-09-14,2024-12-03,...,0.0,0.0,0.0,0.000000,0.0,1.000000,0.0,0.0,0.0,0.000000
2,20170328.0,20180911.0,19640619,0,0,0,0,2018-05-27,2017-03-28,2018-09-11,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000
3,20190508.0,20210930.0,19581209,0,0,0,0,2020-12-26,2019-05-08,2021-09-30,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000
4,20220531.0,20230620.0,19680501,0,0,0,0,2022-12-21,2022-05-31,2023-06-20,...,0.0,0.0,0.0,0.000000,0.0,0.371429,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1050,20211001.0,20230421.0,19760529,0,0,0,0,2023-01-07,2021-10-01,2023-04-21,...,0.0,0.0,0.0,0.881119,0.0,0.125874,0.0,0.0,0.0,0.000000
1051,20150610.0,20170607.0,19710214,0,0,0,0,2017-04-18,2015-06-10,2017-06-07,...,0.0,0.0,0.0,0.000000,0.0,0.000000,1.0,0.0,0.0,0.000000
1052,20180824.0,20200622.0,19731109,0,0,0,0,2020-06-08,2018-08-24,2020-06-22,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000
1053,20230425.0,20260331.0,19700303,0,0,0,0,2024-07-14,2023-04-25,2026-03-31,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000


In [129]:
modules = build_feature_modules(
    snapshot_df,
    module_config
)

In [130]:
allModules = [
        "demographic",
        "experience",
        "organisational",
        "work_history",
        "client_exposure",
        "assignment_stability",

        # NEW
        "temporal_dynamics",
        "caregiver_client_fit",

        "complete"
    ]

In [131]:
runner_config_365 = ModelRunnerConfig(

    # turnover within 12 months
    classification_horizon_days=365,

    # preferable for thesis
    split_mode="temporal",

    test_size=0.20,

    temporal_column="snapshot_date",

    random_state=42,

    include_modules=allModules
)

In [132]:
model_results_365 = run_models(
    snapshot_df,
    modules,
    runner_config_365
)


MODULE: demographic
Features: 7

MODULE: experience
Features: 12

MODULE: organisational
Features: 2

MODULE: work_history
Features: 14

MODULE: client_exposure
Features: 5

MODULE: assignment_stability
Features: 4

MODULE: temporal_dynamics
Features: 18

MODULE: caregiver_client_fit
Features: 27

MODULE: complete
Features: 89


In [133]:
results_365 = model_results_365["results"]

results_365

,module,model,model_type,n_features,n_train,n_test,events_train,events_test,horizon_days,roc_auc,pr_auc,precision,recall,f1,brier,c_index
0,demographic,Logistic Regression,classification,6,844,211,351,161,365.0,0.494037,0.744451,0.752381,0.490683,0.593985,0.255836,NaN
1,demographic,Random Forest,classification,6,844,211,351,161,365.0,0.493478,0.767563,0.742857,0.484472,0.586466,0.287004,NaN
2,demographic,Cox PH,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.526647
3,demographic,Random Survival Forest,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.513912
4,experience,Logistic Regression,classification,12,844,211,351,161,365.0,0.504286,0.770072,0.745455,0.509317,0.605166,0.252288,NaN
5,experience,Random Forest,classification,12,844,211,351,161,365.0,0.487019,0.783230,0.741071,0.515528,0.608059,0.255233,NaN
6,experience,Cox PH,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.505452
7,experience,Random Survival Forest,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.492377
8,organisational,Logistic Regression,classification,2,844,211,351,161,365.0,0.484161,0.757447,0.800000,0.024845,0.048193,0.252278,NaN
9,organisational,Random Forest,classification,2,844,211,351,161,365.0,0.484161,0.757447,0.800000,0.024845,0.048193,0.251850,NaN


In [134]:
runner_config_183 = ModelRunnerConfig(

    # turnover within 12 months
    classification_horizon_days=183,

    # preferable for thesis
    split_mode="temporal",

    test_size=0.20,

    temporal_column="snapshot_date",

    random_state=42,

    include_modules=[
        "demographic",
        "experience",
        "organisational",
        "work_history",
        "client_exposure",
        "assignment_stability",
        "complete"
    ]
)

In [135]:
model_results_183 = run_models(
    snapshot_df,
    modules,
    runner_config_183
)


MODULE: demographic
Features: 7

MODULE: experience
Features: 12

MODULE: organisational
Features: 2

MODULE: work_history
Features: 14

MODULE: client_exposure
Features: 5

MODULE: assignment_stability
Features: 4

MODULE: complete
Features: 89


In [137]:
results_183 = model_results_183["results"]

results_183

,module,model,model_type,n_features,n_train,n_test,events_train,events_test,horizon_days,roc_auc,pr_auc,precision,recall,f1,brier,c_index
0,demographic,Logistic Regression,classification,6,844,211,207,98,183.0,0.569623,0.523684,0.516484,0.479592,0.497354,0.247925,NaN
1,demographic,Random Forest,classification,6,844,211,207,98,183.0,0.598158,0.558734,0.558442,0.438776,0.491429,0.249388,NaN
2,demographic,Cox PH,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.526647
3,demographic,Random Survival Forest,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.513912
4,experience,Logistic Regression,classification,12,844,211,207,98,183.0,0.572783,0.557556,0.505376,0.479592,0.492147,0.244897,NaN
5,experience,Random Forest,classification,12,844,211,207,98,183.0,0.556168,0.546642,0.488889,0.448980,0.468085,0.246669,NaN
6,experience,Cox PH,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.505452
7,experience,Random Survival Forest,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.492377
8,organisational,Logistic Regression,classification,2,844,211,207,98,183.0,0.457874,0.447202,0.200000,0.010204,0.019417,0.253089,NaN
9,organisational,Random Forest,classification,2,844,211,207,98,183.0,0.457874,0.447202,0.200000,0.010204,0.019417,0.253169,NaN


In [ ]:
runner_config_730 = ModelRunnerConfig(

    # turnover within 12 months
    classification_horizon_days=730,

    # preferable for thesis
    split_mode="temporal",

    test_size=0.20,

    temporal_column="snapshot_date",

    random_state=42,

    include_modules=[
        "demographic",
        "experience",
        "organisational",
        "work_history",
        "client_exposure",
        "assignment_stability",
        "complete"
    ]
)

In [ ]:
model_results_730 = run_models(
    snapshot_df,
    modules,
    runner_config_730
)


MODULE: demographic
Features: 7

MODULE: experience
Features: 12

MODULE: organisational
Features: 2

MODULE: work_history
Features: 15

MODULE: client_exposure
Features: 10

MODULE: assignment_stability
Features: 4

MODULE: complete
Features: 103


In [ ]:
results_730 = model_results_730["results"]

results_730

,module,model,model_type,n_features,n_train,n_test,events_train,events_test,horizon_days,roc_auc,pr_auc,precision,recall,f1,brier,c_index
0,demographic,Logistic Regression,classification,6,844,211,535,209,730.0,0.753589,0.997285,1.000000,0.535885,0.697819,0.237874,NaN
1,demographic,Random Forest,classification,6,844,211,535,209,730.0,0.581340,0.992189,0.992481,0.631579,0.771930,0.218033,NaN
2,demographic,Cox PH,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.526647
3,demographic,Random Survival Forest,survival,6,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.513912
4,experience,Logistic Regression,classification,12,844,211,535,209,730.0,0.395933,0.987038,0.990909,0.521531,0.683386,0.241044,NaN
5,experience,Random Forest,classification,12,844,211,535,209,730.0,0.589713,0.993717,0.992424,0.626794,0.768328,0.221698,NaN
6,experience,Cox PH,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.505452
7,experience,Random Survival Forest,survival,12,844,211,844,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.492377
8,organisational,Logistic Regression,classification,2,844,211,535,209,730.0,0.526316,0.991020,1.000000,0.052632,0.100000,0.250359,NaN
9,organisational,Random Forest,classification,2,844,211,535,209,730.0,0.526316,0.991020,1.000000,0.052632,0.100000,0.248186,NaN
